# Clase 9 — RAG Lab con PDFs (solución)

Flujo: PDF → extracción → metadata → chunking → embeddings Cohere → Chroma → retrieval → RAG → evaluación.


In [ ]:
import os, re, sys
from pathlib import Path
import pandas as pd, chromadb, cohere
from dotenv import load_dotenv
from pypdf import PdfReader

load_dotenv()
API_KEY=os.getenv("COHERE_API_KEY")
co=cohere.ClientV2(api_key=API_KEY) if API_KEY else None
print("Python:",sys.executable,"| Cohere:","OK" if co else "Falta COHERE_API_KEY")


## 1. Carga de PDFs y metadata


In [ ]:
PDF_FOLDER=Path("rag_lab_pdfs")
if not PDF_FOLDER.exists(): raise FileNotFoundError(f"No existe {PDF_FOLDER.resolve()}")
pdf_files=sorted(PDF_FOLDER.glob("*.pdf"))
print("PDFs:",len(pdf_files),*[p.name for p in pdf_files],sep="\n- ")

def infer_metadata(filename):
    f=filename.lower(); y=re.search(r"(20\d{2})",f)
    if "vacaciones" in f or "licencias" in f: dept="RRHH"
    elif "reintegro" in f: dept="Finanzas"
    elif any(x in f for x in ("home_office","beneficios","onboarding")): dept="People"
    else: dept="General"
    return {"year":int(y.group(1)) if y else 0,"department":dept,"country":"AR"}

documents=[]
for p in pdf_files:
    r=PdfReader(str(p)); pages=[]
    for n,page in enumerate(r.pages,1): pages.append({"page":n,"text":page.extract_text() or ""})
    documents.append({"source":p.name,"pages":pages,"metadata":infer_metadata(p.name)})
print("Documentos:",len(documents),"| Páginas:",sum(len(d["pages"]) for d in documents))
if documents and documents[0]["pages"]: print(documents[0]["source"],"\n",documents[0]["pages"][0]["text"][:500])


## 2. Chunking


In [ ]:
def clean_text(text): return re.sub(r"\s+"," ",text or "").strip()

def simple_chunk(text,chunk_size=600,overlap=100):
    if chunk_size<=0 or overlap<0 or overlap>=chunk_size: raise ValueError("Revisa chunk_size/overlap")
    text=clean_text(text); out=[]; step=chunk_size-overlap
    for start in range(0,len(text),step):
        x=text[start:start+chunk_size].strip()
        if x: out.append(x)
        if start+chunk_size>=len(text): break
    return out

def build_chunks(documents,chunk_size=600,overlap=100):
    out=[]
    for d in documents:
        md=d["metadata"]
        for p in d["pages"]:
            for i,text in enumerate(simple_chunk(p["text"],chunk_size,overlap)):
                emb=f"Documento: {d['source']}\nAño: {md['year']}\nÁrea: {md['department']}\nPaís: {md['country']}\nPágina: {p['page']}\nContenido: {text}"
                out.append({"id":f"{d['source']}::p{p['page']}::c{i}","source":d["source"],"page":p["page"],"chunk_index":i,**md,"text":text,"embedding_text":emb})
    return out

chunks=build_chunks(documents); print("Chunks:",len(chunks))


## 3. Embeddings + Chroma


In [ ]:
EMBED_MODEL="embed-multilingual-v3.0"
def need_co():
    if co is None: raise RuntimeError("Configura COHERE_API_KEY en .env")
def floats(r):
    v=getattr(r.embeddings,"float",None) or getattr(r.embeddings,"float_",None)
    if v is None: raise RuntimeError("Cohere no devolvió float embeddings")
    return v
def embed_documents(texts,batch_size=96):
    need_co(); out=[]
    for i in range(0,len(texts),batch_size):
        r=co.embed(texts=texts[i:i+batch_size],model=EMBED_MODEL,input_type="search_document",embedding_types=["float"]); out.extend(floats(r))
    return out
def embed_query(q):
    need_co(); r=co.embed(texts=[q],model=EMBED_MODEL,input_type="search_query",embedding_types=["float"]); return floats(r)[0]

embeddings=embed_documents([x["embedding_text"] for x in chunks]) if chunks else []
client=chromadb.Client(); NAME="pia_rag_lab"
try: client.delete_collection(NAME)
except Exception: pass
try: collection=client.create_collection(NAME,configuration={"hnsw":{"space":"cosine"}})
except TypeError: collection=client.create_collection(NAME,metadata={"hnsw:space":"cosine"})
if chunks:
    collection.add(ids=[x["id"] for x in chunks],embeddings=embeddings,documents=[x["text"] for x in chunks],metadatas=[{k:x[k] for k in ("source","page","chunk_index","year","department","country")} for x in chunks])
print("Indexados:",collection.count(),"| Dim:",len(embeddings[0]) if embeddings else 0)


## 4. Retrieval


In [ ]:
def retrieve(query,k=3,where=None):
    cols=["rank","source","page","year","department","distance","text"]
    if collection.count()==0: return pd.DataFrame(columns=cols)
    kw={"query_embeddings":[embed_query(query)],"n_results":min(k,collection.count()),"include":["documents","metadatas","distances"]}
    if where is not None: kw["where"]=where
    r=collection.query(**kw); rows=[]
    for rank,(text,md,dist) in enumerate(zip(r["documents"][0],r["metadatas"][0],r["distances"][0]),1):
        rows.append({"rank":rank,"source":md.get("source"),"page":md.get("page"),"year":md.get("year"),"department":md.get("department"),"distance":dist,"text":text})
    return pd.DataFrame(rows)

q="¿Cuántos días de vacaciones tiene una persona con 7 años de antigüedad?"
display(retrieve(q,5))
for k in (1,3,5): print("K=",k); display(retrieve(q,k))


## 5. Generación baseline y grounded


In [ ]:
GEN_MODEL="command-a-03-2025"
def build_context(df):
    return "\n\n---\n\n".join(f"[Fuente: {r.source} | Página: {r.page} | Año: {r.year} | Área: {r.department}]\n{r.text}" for r in df.itertuples(index=False))
def chat_text(r): return "\n".join(x.text for x in (getattr(getattr(r,"message",None),"content",None) or []) if getattr(x,"text",None)).strip()
def generate_baseline(query,k=3):
    need_co(); rr=retrieve(query,k); ctx=build_context(rr)
    r=co.chat(model=GEN_MODEL,messages=[{"role":"user","content":f"Contexto recuperado:\n{ctx}\n\nPregunta: {query}\nResponde usando el contexto."}],temperature=0)
    return {"answer":chat_text(r),"retrieval":rr}
def generate_grounded(query,k=3,where=None):
    need_co(); rr=retrieve(query,k,where); ctx=build_context(rr)
    sysmsg="Eres un asistente RAG de políticas internas. Usa únicamente la documentación recuperada; no uses conocimiento externo. Prioriza la versión más reciente. Si no alcanza la evidencia responde: No encuentro evidencia suficiente en la documentación recuperada. Cita fuente y página. Si hay contradicciones, explica la diferencia y los años."
    r=co.chat(model=GEN_MODEL,messages=[{"role":"system","content":sysmsg},{"role":"user","content":f"DOCUMENTACIÓN RECUPERADA\n{ctx or '[Sin resultados]'}\n\nPREGUNTA\n{query}"}],temperature=0)
    return {"answer":chat_text(r),"retrieval":rr}

b=generate_baseline(q); g=generate_grounded(q); print("BASELINE\n",b["answer"],"\n\nGROUNDED\n",g["answer"]); display(g["retrieval"])


## 6. Fuera de contexto + filtros


In [ ]:
for q2 in ("¿Cuál es la capital de Francia?","¿Quién es Harry Potter?","¿Cuál es el sueldo de Ana Pérez?"):
    print("\n",q2); display(retrieve(q2,3)); print(generate_grounded(q2,3)["answer"])
vq="¿Cuántos días de vacaciones corresponden entre 5 y 10 años?"
print("SIN FILTRO"); display(retrieve(vq,5))
print("YEAR=2026"); display(retrieve(vq,5,where={"year":2026}))


## 7. Evaluación Hit@K


In [ ]:
evaluation_set=[
 {"query":"¿Cuántos días de vacaciones tiene alguien con 7 años de antigüedad?","expected_source":"politica_vacaciones_2026.pdf"},
 {"query":"¿Con cuánta anticipación debo pedir vacaciones?","expected_source":"politica_vacaciones_2026.pdf"},
 {"query":"¿Cuántos días por semana puedo trabajar remoto?","expected_source":"politica_home_office_2026.pdf"}]
def evaluate_retrieval(items,k=3):
    rows=[]
    for x in items:
        r=retrieve(x["query"],k); mm=r[r["source"]==x["expected_source"]]; hit=not mm.empty
        rows.append({"query":x["query"],"expected_source":x["expected_source"],"k":k,"hit":hit,"rank":int(mm.iloc[0]["rank"]) if hit else None})
    return pd.DataFrame(rows)
summary=[]
for k in (1,3,5):
    r=evaluate_retrieval(evaluation_set,k); hr=r["hit"].mean() if len(r) else 0; summary.append({"k":k,"hit_rate":hr}); print(f"K={k} Hit@K={hr:.2%}"); display(r)
pd.DataFrame(summary)


## 8. Comparación de chunk size


In [ ]:
rows=[]
for size,ov in ((300,50),(600,100),(1000,150)):
    cc=build_chunks(documents,size,ov); lens=[len(x["text"]) for x in cc]
    rows.append({"chunk_size":size,"overlap":ov,"num_chunks":len(cc),"avg_chunk_length":sum(lens)/len(lens) if lens else 0,"min_chunk_length":min(lens) if lens else 0,"max_chunk_length":max(lens) if lens else 0})
pd.DataFrame(rows)
